In [1]:
from glob import glob

for g in glob("../data/*.pdf"):
    print(g)

../data\2040_seoul_plan.pdf
../data\OneNYC_2050_StrategicPlan.pdf


read_pdf_and_split_text 함수 만들기


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def read_pdf_and_split_text(pdf_path, chunk_size=1000, chunk_overlap=100):
    """
    주어진 PDF 파일을 읽고 텍스트를 분할합니다.
    매개변수:
      pdf_path (str): PDF 파일의 경로.
      chunk_size (int, 선택적): 각 텍스트 청크의 크기, 기본값은 1000 입니다.
      chunk_overlap (int, 선택적): 청크 간의 중첩 크기. 기본값은 100 입니다.
    반환값:
      list: 분할된 텍스트 청크의 리스트.
    """

    print(f"PDF: {pdf_path} -----------------------------------")

    pdf_loader = PyPDFLoader(pdf_path)
    data_from_pdf = pdf_loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )

    splits = text_splitter.split_documents(data_from_pdf)

    print(f"Number of splits: {len(splits)}\n")
    return splits

C:\Users\bny64\AppData\Local\Temp\ipykernel_10384\70338277.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from glob import glob
import os
import time
import dotenv

dotenv.load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# 임베딩 모델 선언
embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2", google_api_key=GEMINI_API_KEY
)

persist_directory = "../chroma_store"

if os.path.exists(persist_directory):
    print("Loading existing Chroma store")
    vectorstore = Chroma(
        persist_directory=persist_directory, embedding_function=embedding
    )

else:
    print("Creating new Chroma store")
    # 429 Resource Exhausted 방지를 위해 청크 단위로 나누어 인덱싱을 수행합니다.
    vectorstore = None
    batch_size = 40

    for g in glob("../data/*.pdf"):
        chunks = read_pdf_and_split_text(g)
        for i in range(0, len(chunks), batch_size):
            print(
                f"Adding documents from {i} to {min(i + batch_size, len(chunks))} for {g}..."
            )
            if vectorstore is None:
                # 첫 번째 청크로 vectorstore 초기화
                vectorstore = Chroma.from_documents(
                    documents=chunks[i : i + batch_size],
                    embedding=embedding,
                    persist_directory=persist_directory,
                )
            else:
                vectorstore.add_documents(documents=chunks[i : i + batch_size])
            time.sleep(60)
    print("Chroma store creation completed!")

Loading existing Chroma store


In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

chunks = retriever.invoke("서울 온실가스 저감 계획")

for chunk in chunks:
    print(chunk.metadata)
    print(chunk.page_content)

{'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2023-02-14T18:21:42+09:00', 'page_label': '71', 'author': '', 'total_pages': 272, 'title': '', 'source': '../data\\2040_seoul_plan.pdf', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creationdate': '2023-02-14T11:05:36+09:00', 'page': 70}
- 도시 기반시설 부문(건물): 무공해 빌딩 확대
- 도시 기반시설 부문(교통): 무공해 차량(ZEV) 보급 촉진
- 자원/산업 부문: 3R(줄이기, 재사용, 재활용), 플라스틱, 음식물쓰레기, HFC배출량 감소
- 기후변화 적응: 기후변화 적응 조치 강화
- 거버넌스: 기업(민간), 지자체, 주요 도시 등과 협력, 재생에너지 사업 투자 촉진
8) https://fpcj.jp/en/prlisting/tokyo_20211012/
9) https://zenbird.media/zero-emissions-tokyo-an-ambitious-climate-change-strategy/
{'total_pages': 272, 'source': '../data\\2040_seoul_plan.pdf', 'author': '', 'page': 151, 'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2023-02-14T18:21:42+09:00', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'page_label': '152', 'creationdate': '2023-02-14T11:05:36+09:00', 'title': ''}
3. 부문별 전략계획
3.1 추진위원회 회의 결과
3.2 서울시 관련 실·국·본부 의견
{'producer': 'Acrobat Distille

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from typing import Literal
from pydantic import BaseModel, Field


# Data model
class RouteQuery(BaseModel):
    """사용자 쿼리를 가장 관련성이 높은 데이터 소스로 라우팅합니다."""

    datasource: Literal["vectorstore", "casual_talk"] = Field(
        ...,
        description="""
    사용자 질문에 따라 casual_talk 또는 vectorstore로 라우팅합니다.
    - casual_talk: 일상 대화를 위한 데이터 소스, 사용자가 질문을 할 때 사용합니다.
    - vectorstore: 사용자 질문에 답하기 위해 RAG로 vectorstore 검색이 필요한 경우 사용합니다.
    """,
    )

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv()

# 모델 정의
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite", google_api_key=os.getenv("GEMINI_API_KEY")
)


# 특정 모델을 structured output(구조화된 출력)과 함께 사용하기 위해 설정
structured_llm_router = model.with_structured_output(RouteQuery)

router_system = """
당신은 사용자의 질문을 vectorstore 또는 casual_talk으로 라우팅하는 전문가입니다.
- vectorstore에는 서울, 뉴욕의 발전계획과 관련된 문서가 포함되어 있습니다. 이 주제에 대한 질문에는 vectorstore를 사용하십시오.
- 사용자의 질문이 일상 대화와 관련된 경우 casual_talk을 사용하십시오.
"""

# 시스템 메시지와 사용자의 질문을 포함하는 프롬프트 템플릿 생성
route_prompt = ChatPromptTemplate.from_messages(
    [("system", router_system), ("human", "{question}")]
)

# 라우터 프롬프트와 구조화된 출력 모델을 결합한 객체
question_router = route_prompt | structured_llm_router

In [5]:
print(question_router.invoke({"question": "서울 온실가스 저감 계획은 무엇인가요?"}))

print(question_router.invoke({"question": "잘 지냈어?"}))

datasource='vectorstore'
datasource='casual_talk'


In [14]:
from langchain_core.prompts import PromptTemplate


class GradeDocuments(BaseModel):
    """검색된 문서가 질문과 관련성이 있는지 yes 또는 no로 평가합니다."""

    binary_score: Literal["yes", "no"] = Field(
        description="문서가 질문과 관련이 있는지 여부를 'yes' 또는 'no'로 평가합니다."
    )


structured_llm_grader = model.with_structured_output(GradeDocuments)

ChatPromptTemplate: 기존 대화내용이 이어질 경우 챗봇에서 많이 사용
PromptTemplate: 기존 대화 내용이 이어질 필요가 없는 경우

In [ ]:
grader_prompt = PromptTemplate.from_template(
    """
당신은 검색된 문서가 사용자 질문과 관련이 있는지 평가하는 평가자입니다. \n
문서에 사용자 질문과 관련된 키워드 또는 의미가 포함되어 있으면, 해당 문서를 관련성이 있다고 평가하십시오. \n
엄격한 테스트가 필요하지 않습니다. 목표는 잘못된 검색 결과를 걸러내는 것입니다. \n
문서가 질문과 관련이 있는지 여부를 나타내기 위해 'yes' 또는 'no' 로 이진 점수를 부여하십시오. 

Retrieved document: \n {document} \n\n
User question: {question}
"""
)

retriever_grader = grader_prompt | structured_llm_grader
question = "서울시 자율주행 관련 계획"
documents = retriever.invoke(question)

for doc in documents:
    print("---------------")
    print(doc)

---------------
page_content='104 2. 시민참여와 미래상
부문 주요 계획과제 세부내용
교통
개인형 교통수단 활성화
전동 퀵 보드, 자전거 등 전용 도로 확보
자전거 도로 개선
공유 모빌리티 규제 완화
대중교통 편의 증진
쾌적하게 이용할 수 있는 대중교통 확충
심야버스 확충, 버스 총량제 폐지
고밀보다 충분한 교통 기반시설 마련
지하철역사 내 공간을 문화, 녹지 시설로 조성
보행친화도시 추구
골목 비우기 실천을 통한 총량 확보
보행친화도시로 개발
무 장 애  보 행 로  설 치
4대문 안 자가 차량 진입 금지
친환경 교통 정책 강화
CO2, 미세먼지 감소 위한 환경 규제
수소·전기차 충전시설 확충
차량 2부제 규제 강화
자치구별 녹색교통 개선 및 확대
노후 디젤차 서울 진입 금지
남북
협력 남북 교통 인프라 확충 고속철 포함 남북 교통 인프라 확충' metadata={'creator': 'PScript5.dll Version 5.2.2', 'title': '', 'moddate': '2023-02-14T18:21:42+09:00', 'page_label': '115', 'creationdate': '2023-02-14T11:05:36+09:00', 'page': 114, 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'total_pages': 272, 'author': '', 'source': '../data\\2040_seoul_plan.pdf'}
---------------
page_content='5만 그루의 나무를 심는 등 게이트 재개발이 포함된 계획으로,6) 파리시는 2024년에서 2030년까지의 
로드맵을 수립하여 단계적으로 전환할 예정임
- (1단계 - 2024년) 대중교통과 자율주행자동차 전용도로 조성, 속도 50km/h 제한, 화물운송차량
(3.5t 초과) 진입 금지, 소음방지 장치 개발, 대체 교통수단, 버스 네트워크 개발 등
- (2단계 - 2030년) 특정 유형의 차량 전용

In [17]:
# 가져온 청크 중 서울시 자율주행 계획과 관련된 문서만 찾아 filtered_docs에 담는다.
filtered_docs = []

for i, doc in enumerate(documents):
    print(f"Document {i + 1}:")
    is_relevant = retriever_grader.invoke(
        {"question": question, "document": doc.page_content}
    )
    print(is_relevant)
    print(doc.page_content[:200])
    print("====================\n\n")

    if is_relevant.binary_score == "yes":
        filtered_docs.append(doc)

print(f"Filtered documents: {len(filtered_docs)}")

Document 1:
binary_score='no'
104 2. 시민참여와 미래상
부문 주요 계획과제 세부내용
교통
개인형 교통수단 활성화
전동 퀵 보드, 자전거 등 전용 도로 확보
자전거 도로 개선
공유 모빌리티 규제 완화
대중교통 편의 증진
쾌적하게 이용할 수 있는 대중교통 확충
심야버스 확충, 버스 총량제 폐지
고밀보다 충분한 교통 기반시설 마련
지하철역사 내 공간을 문화, 녹지 시설로 조성
보행친화도


Document 2:
binary_score='no'
5만 그루의 나무를 심는 등 게이트 재개발이 포함된 계획으로,6) 파리시는 2024년에서 2030년까지의 
로드맵을 수립하여 단계적으로 전환할 예정임
- (1단계 - 2024년) 대중교통과 자율주행자동차 전용도로 조성, 속도 50km/h 제한, 화물운송차량
(3.5t 초과) 진입 금지, 소음방지 장치 개발, 대체 교통수단, 버스 네트워크 개발 등
- (2단


Document 3:
binary_score='no'
 스마트기술의 발달로 시민참여 방안이 확대되었으며, 한정된 시간(단기 설문조사 등)이 아닌 꾸준하게 
시민의 의견을 받고 상호소통(피드백) 방식 필요
미래 서울에 대한 다양한 예측과 정보를 제공, 시민이 주체적으로 참여할 수 있는 환경 마련
 형식적인 참여 독려가 아닌 정보제공을 통한 자발적인 참여 환경 조성이 중요함


Document 4:
binary_score='yes'
3.2 서울시 관련 실·국·본부 의견 169
❚ 회의 결과
안전총괄과의 주요 계획
 [안전도시 서울도시기본계획(2018)] 서울시 재난 및 안전관리기본법
 [안전관리계획] 연례 법정계획
 [서울도시회복력강화계획] 비법정 중기계획으로 대형 재난 및 사고 발생 시 최대한 빠르게 정상 
궤도로 복귀할 수 있도록 하자는 근본적인 철학이 담긴 광범위한 계획
2


Document 5:
binary_score='no'
4. 공간계획
4.1 서울시 관련 실·국·본부 의견
4.2 자치구 의견
4.3 공간계획 자문회의 결과


In [18]:
### Generate
# PromptTemplate을 사용해 RAG를 위한 프롬프트 생성

rag_generate_system = """
너는 사용자의 질문에 대해 주어진 context에 기반하여 답변하는 도시 계획 전문가이다.
주어진 context는 vectorstore에서 검색된 결과이다.
주어진 context를 기반으로 사용자의 question에 대해 답변하라.

===================================
question: {question}
context: {context}
"""

# PromptTemplate을 생성해 question과 context를 포매팅
rag_prompt = PromptTemplate(
    input_variables=["question", "context"], template=rag_generate_system
)

# rag chain
rag_chain = rag_prompt | model

question = "서울시 자율주행 관련 계획"

rag_chain.invoke({"question": question, "context": filtered_docs})

AIMessage(content=[{'type': 'text', 'text': "도시 계획 전문가로서 제공된 문서를 바탕으로 서울시 자율주행 관련 계획에 대해 답변드립니다.\n\n제공된 '2040 서울도시기본계획' 관련 회의 자료에 따르면, 서울시의 자율주행과 관련된 현황 및 계획적 과제는 다음과 같습니다.\n\n1. **미래 통행 변화에 대한 고려 필요성**: 현재 서울시의 중장기 계획 수립 과정에서 20년 후 서울 시민의 통행 환경 변화를 야기할 '자율주행차' 도입에 대한 고려가 여전히 부족하다는 점이 지적되었습니다.\n2. **중장기 예측의 중요성**: 도시 계획의 실효성을 확보하기 위해서는 인구 구조나 기술 변화 등을 반영해야 합니다. 특히 자율주행뿐만 아니라 카셰어링과 같은 새로운 이동수단의 변화를 감안한 '자동차 통행량 중장기 예측'이 필수적이라는 의견이 제시되었습니다.\n3. **스마트 기술 및 자동화 도입**: 안전 부문과 관련하여, 기존의 육안 점검 위주였던 안전점검 체계를 개선하기 위해 스마트 기술을 적용한 정보화 및 자동화를 추진해야 한다는 점이 강조되었습니다.\n\n결론적으로, 현재 서울시는 자율주행 등 미래 기술이 가져올 통행 패턴의 변화를 도시 계획에 보다 적극적으로 반영하고, 이를 기반으로 한 실효성 있는 중장기 예측 체계를 마련해야 하는 과제를 안고 있습니다.", 'extras': {'signature': 'EnEKbwERTTIPQRSNTIZXcPHr1+Ld3pSfOlK4quauaTq7PkgjelfkgcFNfU/K+jKi4PvY/58wf1UKwehK/LJbTDXGkFY6Ql2qYCuiRXrCyccyGXfBZaGrVZzX5uLeqoV+EpnRVHVrhyfeS8OdFp2ITsMq1w=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_pr

In [19]:
from typing import List
from typing_extensions import TypedDict


class GraphState(TypedDict):
    question: str  # 사용자 질문
    generation: str  # LLM 생성 결과
    documents: List[str]  # 검색된 문서

In [ ]:
def route_question(state):
    """
    사용자 질문을 vectorstore 또는 casual_talk로 라우팅합니다.

    Args:
        state (dict): 현재 graph state

    return:
        state (dict): 라우팅된 데이터 소스와 사용자 질문을 포함하는 새로운 graph state
    """
    print("-----ROUTE-----")
    question = state["question"]
    route = question_router.invoke({"question": question})

    print(f"---Routing to {route.datasource} ---")
    return route.datasource